In this notebook, we will use differents machine learning algorithms to predict whether a song will be a hit or not based on its features. We will evaluate the performance of each model and compare their results.

We approach this problem from both perspectives: regression (predicting the exact popularity score), binary classification (hit vs non-hit), and multi-class classification(multiple popularity levels). By exploring various algorithms and tuning their hyperparameters, we aim to identify the most effective model for modeling song popularity.

Import necessary libraries and modules

In [ ]:
import sys

sys.path.append("..")  # To allow imports from src directory

from src.trainer import train_model, evaluate_model, report_performance
from src.utils import load_and_preprocess_data

We have already prepared some helpers functions in the `src/utils.py` file to load and preprocess the data as well as those in `src/models.py` to create and evaluate different models.

In the `load_and_preprocess_data` function from `src/utils.py`, we load the dataset, preprocess the features, and split the data into training and testing sets. The function returns the processed training and testing feature sets, the corresponding target values, and the feature names for interpretability. For the preprocessing, we divide the features into three categories:
- Numerical features: These features are scaled using `sklearn.preprocessing.StandardScaler` to ensure they have a mean of 0 and a standard deviation of 1.
- Categorical features: These features are encoded using `sklearn.preprocessing.OneHotEncoder` to convert categorical variables into a format that can be provided to machine learning algorithms.
- Track age feature: This feature is transformed by calculating the age of the track based on its release year.

Below are the parameters used for loading and preprocessing the data:

In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42
SAMPLE_SIZE = 0.01

# Due to the nature of the dataset and to save time during execution, we will disable hyperparameter tuning for now
ENABLE_HYPERPARAMETER_TUNING = False

## 1. Preliminary Analysis

In this section, we will experiment with different baseline machine learning models to see how well they perform on our dataset.

### 1.1 Regression

Due to the nature of one-hot encoding in our preprocessing code, loading all samples of data will lead to extremely high memory usage (> 16 GB of RAM). Therefore, we will limit the number of samples loaded for this demonstration as a fraction of 0.01 (1% ~ 16,000 samples) of the total dataset.

For reference, we have approximately 1.6 million rows in the dataset, when apply one-hot encoding on the `genre` feature which has more than 3000 unique values. This results in an additional 1.6M x 3000 = 4.8 billion entries in the feature matrix, with each entry store in `float` format (4 bytes), leading to a memory requirement of over 19 GB just for this feature alone.

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_and_preprocess_data(
    "../datasets/spotify_tracks_preprocessed.csv",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    problem_type="regression",
    sample_size=SAMPLE_SIZE,
    use_categorical=True,
)

In [ ]:
# Define hyperparameters for different models

# Parameters for Ridge Regression
ridge_params = {
    "alpha": 1.0,  # Regularization strength; default is 1.0
}

# Parameters for K-Nearest Neighbors (KNN)
knn_params = {
    "n_neighbors": 5,  # Number of neighbors for the default model; default is 5
}

# Parameters for Random Forest
rf_params = {
    "n_estimators": 50,  # Default number of trees; default is 50
}

if ENABLE_HYPERPARAMETER_TUNING:
    ridge_params["param_grid"] = {
        "alpha": [0.1, 1.0, 10.0]  # Test different regularization strengths
    }
    knn_params["param_grid"] = {
        "n_neighbors": [3, 5, 7],  # Test different numbers of neighbors
        "weights": ["uniform", "distance"],  # Test uniform vs. distance-based weighting
    }
    rf_params["param_grid"] = (
        {
            "n_estimators": [25, 50, 100],  # Test different numbers of trees
            "max_depth": [None, 5, 10],  # Test different maximum tree depths
            "min_samples_split": [2, 5],  # Test minimum samples required to split a node
        },
    )

In [ ]:
# Train dffiferent models
ridge_model = train_model(X_train, y_train, "regression", "ridge", ridge_params)
knn_model = train_model(X_train, y_train, "regression", "knn", knn_params)
rf_model = train_model(X_train, y_train, "regression", "random_forest", rf_params)

In [ ]:
# Evaluate models
ridge_metrics = evaluate_model(ridge_model, X_test, y_test, "regression")
knn_metrics = evaluate_model(knn_model, X_test, y_test, "regression")
rf_metrics = evaluate_model(rf_model, X_test, y_test, "regression")

In [ ]:
print("Ridge result:")
report_performance(ridge_metrics)
print("KNN result:")
report_performance(knn_metrics)
print("Random Forest result:")
report_performance(rf_metrics)

**Random Forest** proved to be the superior model ($R^2 \approx 0.45$), outperforming the others by effectively capturing non-linear interactions between audio features, though **Ridge Regression** was a surprisingly strong runner-up ($R^2 \approx 0.41$), indicating that simple linear factors like `track_age` are the primary drivers of popularity.
<!-- **KNN** failed significantly ($R^2 \approx 0.17$) because it could not handle the high dimensionality created by the one-hot encoded genres (the "curse of dimensionality"). -->
Ultimately, the fact that even the best model leaves 55% of the variance unexplained suggests that intrinsic audio features alone are insufficient for prediction, and external factors like artist fame or marketing budget are likely required to improve accuracy further.

### 1.2. Classification

#### 1.2.1 Binary Classification

Since the original `popularity` score is numerical within 0-100 range, we first convert it into a binary classification problem by defining a threshold (e.g., popularity >= 50 as "hit" and < 50 as "non-hit").

Base on Notebook 2 Preprocessing results, we know that P80 is 31 (80% of tracks have a score below 31) and P90 is 41 (~90% of tracks have a score below 41). Therefore, we can also try different thresholds like 31 and 41 to see how the models perform under different definitions of "hit".

##### 1.2.1a Popularity threshold as 80th percentile (31)

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_and_preprocess_data(
    "../datasets/spotify_tracks_preprocessed.csv",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    problem_type="classification",
    popularity_threshold=31,
    sample_size=SAMPLE_SIZE,
    use_categorical=True,
)

In [ ]:
# Parameters for Logistic Regression
logistic_params = {
    "penalty": None,  # Regularization type; default is None, choices are 'l2', 'l1', 'elasticnet', or None
}

# Parameters for K-Nearest Neighbors (KNN)
knn_params = {
    "n_neighbors": 5,  # Number of neighbors for the default model; default is 5
}

# Parameters for Random Forest
rf_params = {
    "n_estimators": 50,  # Default number of trees; default is 50
}

if ENABLE_HYPERPARAMETER_TUNING:
    logistic_params["param_grid"] = {
        "penalty": [None, "l2", "l1", "elasticnet"]  # Test different regularization types
    }
    knn_params["param_grid"] = {
        "n_neighbors": [3, 5, 7],  # Test different numbers of neighbors
        "weights": ["uniform", "distance"],  # Test uniform vs. distance-based weighting
    }
    rf_params["param_grid"] = (
        {
            "n_estimators": [25, 50, 100],  # Test different numbers of trees
            "max_depth": [None, 5, 10],  # Test different maximum tree depths
            "min_samples_split": [2, 5],  # Test minimum samples required to split a node
        },
    )

In [ ]:
# Train dffiferent models
logistic_model = train_model(X_train, y_train, "classification", "logistic_regression", logistic_params)
knn_model = train_model(X_train, y_train, "classification", "knn", knn_params)
rf_model = train_model(X_train, y_train, "classification", "random_forest", rf_params)

In [ ]:
# Evaluate models
logistic_metrics = evaluate_model(logistic_model, X_test, y_test, "classification")
knn_metrics = evaluate_model(knn_model, X_test, y_test, "classification")
rf_metrics = evaluate_model(rf_model, X_test, y_test, "classification")

In [ ]:
print("Logistic result:")
report_performance(logistic_metrics)
print("KNN result:")
report_performance(knn_metrics)
print("Random Forest result:")
report_performance(rf_metrics)

**Logistic Regression** serves as the most effective generalist for this imbalanced **80/20 split**, securing the best **F1-Score (0.53)** and **Recall (43%)**, making it preferable for maximizing the discovery of potential hits compared to **Random Forest**, which acted as a "conservative sniper" with high **Precision (85%)** but missed two-thirds of actual popular tracks. **KNN** lagged significantly behind both. Ultimately, the generally low Recall indicates that the models are biased toward the majority "unpopular" class, necessitating the use of class blancing to force the algorithms to pay more attention to the minority "hit" class.

##### 1.2.1b Popularity threshold as 90th percentile (41)

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_and_preprocess_data(
    "../datasets/spotify_tracks_preprocessed.csv",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    problem_type="classification",
    popularity_threshold=41,
    sample_size=SAMPLE_SIZE,
    use_categorical=True,
)

In [ ]:
# We keep the same hyperparameters as before for consistency
# Train dffiferent models
logistic_model = train_model(X_train, y_train, "classification", "logistic_regression", logistic_params)
knn_model = train_model(X_train, y_train, "classification", "knn", knn_params)
rf_model = train_model(X_train, y_train, "classification", "random_forest", rf_params)

In [ ]:
# Evaluate models
logistic_metrics = evaluate_model(logistic_model, X_test, y_test, "classification")
knn_metrics = evaluate_model(knn_model, X_test, y_test, "classification")
rf_metrics = evaluate_model(rf_model, X_test, y_test, "classification")

In [ ]:
print("Logistic result:")
report_performance(logistic_metrics)
print("KNN result:")
report_performance(knn_metrics)
print("Random Forest result:")
report_performance(rf_metrics)

The severe **90/10 class imbalance** effectively broke the **Random Forest** model; it fell into the "accuracy paradox," achieving high accuracy (~90%) simply by predicting almost every track as "unpopular," resulting in a negligible Recall of roughly 9%. **Logistic Regression** emerged as the only somewhat viable model, identifying significantly more hits (Recall ~23%) and achieving the highest F1-Score (0.32), proving that it handled the sparse minority class better than the other default models. **KNN** performed worse than a baseline guess. This experiment highlights the critical need for class balancing techniques (e.g., oversampling, SMOTE, class weights) to meaningfully improve hit detection in such imbalanced scenarios.

#### 1.2.2 Multi-Class Classification

Based on how popularity works, we can also define multiple classes (e.g., low, medium, high popularity) and train models to predict these categories. Specifically, we can define 3 classes:
- Low Popularity: 0-20
- Medium Popularity: 21-50
- High Popularity: 51-100

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_and_preprocess_data(
    "../datasets/spotify_tracks_preprocessed.csv",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    problem_type="classification",
    popularity_threshold=[20, 50],
    sample_size=SAMPLE_SIZE,
    use_categorical=True,
)

In [ ]:
# Parameters for Ridge Regression
logistic_params = {
    "penalty": None,  # Regularization type; default is None, choices are 'l2', 'l1', 'elasticnet', or None
}

# Parameters for K-Nearest Neighbors (KNN)
knn_params = {
    "n_neighbors": 5,  # Number of neighbors for the default model; default is 5
}

# Parameters for Random Forest
rf_params = {
    "n_estimators": 50,  # Default number of trees; default is 50
}

if ENABLE_HYPERPARAMETER_TUNING:
    logistic_params["param_grid"] = {
        "penalty": [None, "l2", "l1", "elasticnet"]  # Test different regularization types
    }
    knn_params["param_grid"] = {
        "n_neighbors": [3, 5, 7],  # Test different numbers of neighbors
        "weights": ["uniform", "distance"],  # Test uniform vs. distance-based weighting
    }
    rf_params["param_grid"] = (
        {
            "n_estimators": [25, 50, 100],  # Test different numbers of trees
            "max_depth": [None, 5, 10],  # Test different maximum tree depths
            "min_samples_split": [2, 5],  # Test minimum samples required to split a node
        },
    )

In [ ]:
# Train dffiferent models
logistic_model = train_model(X_train, y_train, "classification", "logistic_regression", logistic_params)
knn_model = train_model(X_train, y_train, "classification", "knn", knn_params)
rf_model = train_model(X_train, y_train, "classification", "random_forest", rf_params)

In [ ]:
# Evaluate models
logistic_metrics = evaluate_model(logistic_model, X_test, y_test, "classification")
knn_metrics = evaluate_model(knn_model, X_test, y_test, "classification")
rf_metrics = evaluate_model(rf_model, X_test, y_test, "classification")

In [ ]:
print("Logistic result:")
report_performance(logistic_metrics)
print("KNN result:")
report_performance(knn_metrics)
print("Random Forest result:")
report_performance(rf_metrics)

**Logistic Regression** is definitively the best model for this multiclass problem, achieving the highest **Macro F1-Score (0.55)** and **Recall (0.53)**, proving it is the only algorithm effectively attempting to learn the minority "Mid" and "High" tiers rather than just maximizing accuracy on the majority "Low" tier. While **Random Forest** achieved the highest **Precision (0.67)**, its low Recall (0.48) and inferior Macro score indicate it is too conservative, frequently misclassifying potential hits as "Obscure" to play it safe. The low F1-Score highlights that all models are struggling to distinguish the "Hit" songs (Class 2) from the noise, this requires further techniques like class weighting or oversampling to improve detection of these rare but important tracks.

## 2. Extended Analysis

### 2.1 The importance of categorical features

In this section, we will test the importance of categorial features (e.g., genre) in our models by removing them and observing the performance changes. This will help us understand how much these features contribute to the prediction of song popularity. For this analysis, we will focus on Random Forest model for both regression and classification tasks.

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_and_preprocess_data(
    "../datasets/spotify_tracks_preprocessed.csv",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    problem_type="regression",
    sample_size=SAMPLE_SIZE,
    use_categorical=False,
)

# Train and evaluate model
rf_model = train_model(X_train, y_train, "regression", "random_forest", {"n_estimators": 50})
rf_metrics = evaluate_model(rf_model, X_test, y_test, "regression")
print("Random Forest on Regression without categorical features result:")
report_performance(rf_metrics)

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_and_preprocess_data(
    "../datasets/spotify_tracks_preprocessed.csv",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    problem_type="classification",
    popularity_threshold=31,
    sample_size=SAMPLE_SIZE,
    use_categorical=False,
)

# Train and evaluate model
rf_model = train_model(X_train, y_train, "classification", "random_forest", {"n_estimators": 50})
rf_metrics = evaluate_model(rf_model, X_test, y_test, "classification")
print("Random Forest on Classification without categorical features result:")
report_performance(rf_metrics)

The experiment proves that categorical features contain **Context (Genre/Key)** is the primary driver of popularity, vastly outweighing **Content (Audio Features)**. Removing categorical data caused the model’s explanatory power to collapse by nearly 63% ($R^2$ dropping from 0.45 to 0.17), demonstrating that physical audio properties like tempo or energy are weak predictors in isolation. Furthermore, the classification Recall plummeted from 32% to 15%, indicating that without the market context provided by genre, the model loses the ability to distinguish "Hits" from generic tracks.

### 2.2 Categorical features as vector embeddings

Knowing that categorical features are crucial, we can further improve our models by representing these features as vector embeddings by converting features like `track_name, artist_name, genre` into, for example, "Track: Hello | Artist: Adele | Genre: Pop". And then using a pretrained embedding model to convert these text features into dense vector representations. This approach can capture semantic relationships between different categories better than one-hot encoding.

In this experiment, we will use `MinishLab/potion-base-8M` model due to its computational efficiency. This model generates 256-dimensional embeddings, which are compact yet rich in semantic information, making them suitable for our classification task without overwhelming our computational resources. The generated embeddings will then be concatenated with the numerical features before training the Random Forest model.

In [ ]:
# Run this once to get the text embeddings
from src.utils import generate_and_save_embeddings

generate_and_save_embeddings(
    input_csv="../datasets/spotify_tracks_preprocessed.csv",
    output_parquet="../datasets/text_embeddings.parquet",
    model_name="MinishLab/potion-base-8M",
    device="cpu",
)

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_and_preprocess_data(
    "../datasets/spotify_tracks_preprocessed.csv",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    problem_type="regression",
    sample_size=SAMPLE_SIZE,
    use_categorical=True,
    use_categorical_embeddings=True,
    embeddings_path="../datasets/text_embeddings.parquet",
)

# Train and evaluate model
rf_model = train_model(X_train, y_train, "regression", "random_forest", {"n_estimators": 50})
rf_metrics = evaluate_model(rf_model, X_test, y_test, "regression")
report_performance(rf_metrics)

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_and_preprocess_data(
    "../datasets/spotify_tracks_preprocessed.csv",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    problem_type="classification",
    popularity_threshold=31,
    sample_size=SAMPLE_SIZE,
    use_categorical=True,
    use_categorical_embeddings=True,
    embeddings_path="../datasets/text_embeddings.parquet",
)

# Train and evaluate model
rf_model = train_model(X_train, y_train, "classification", "random_forest", {"n_estimators": 50})
rf_metrics = evaluate_model(rf_model, X_test, y_test, "classification")
report_performance(rf_metrics)

The experiment show that although using vector embeddings from `MinishLab/potion-base-8M` improved upon the "no-categorical" baseline ($R^2 \approx 0.17$) by capturing general semantic clusters (e.g., relating "Rock" to "Metal"), it failed to beat the baseline One-Hot Encoding($R^2 \approx 0.28$ versus $R^2 \approx 0.45$). One possible reason is that One-Hot Encoding provides a more precise and distinct representation of categorical variables, allowing the model to learn specific associations between categories and popularity.

### 2.3 Categorical features as Target Encoding

In this experiment, we will use Target Encoding to encode the categorical features. Target Encoding replaces each category with the mean of the target variable for that category. This approach can capture the relationship between categorical features and the target variable more effectively than one-hot encoding, especially when there are many unique categories.

For example, if we have a categorical feature "genre" with categories "Pop", "Rock", and "Jazz", and the average popularity scores for these genres are 70, 50, and 40 respectively, then we would replace "Pop" with 70, "Rock" with 50, and "Jazz" with 40 in the feature set.

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_and_preprocess_data(
    "../datasets/spotify_tracks_preprocessed.csv",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    problem_type="regression",
    sample_size=SAMPLE_SIZE,
    use_categorical=True,
    categorical_encoding="target_encoding",
)

# Train and evaluate model
rf_model = train_model(X_train, y_train, "regression", "random_forest", {"n_estimators": 50})
rf_metrics = evaluate_model(rf_model, X_test, y_test, "regression")
report_performance(rf_metrics)

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_and_preprocess_data(
    "../datasets/spotify_tracks_preprocessed.csv",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    problem_type="classification",
    popularity_threshold=31,
    sample_size=SAMPLE_SIZE,
    use_categorical=True,
    categorical_encoding="target_encoding",
)

# Train and evaluate model
rf_model = train_model(X_train, y_train, "classification", "random_forest", {"n_estimators": 50})
rf_metrics = evaluate_model(rf_model, X_test, y_test, "classification")
report_performance(rf_metrics)

**Target Encoding** has proven to be the superior strategy, definitively outperforming One-Hot Encoding by breaking the **0.50 $R^2$ barrier** and delivering a massive **18% increase in Recall** (50.5% vs. 32.0%). By condensing the high-cardinality `genre` data into a single, continuous "popularity signal," the Random Forest no longer wastes capacity filtering through thousands of sparse columns, allowing it to aggressively identify potential hits that the conservative One-Hot model missed. Although this increased sensitivity resulted in a slight drop in Precision (~71% vs. 80%), the trade-off is strategically sound: the model has evolved from a conservative classifier that misses most hits into a robust discovery tool that successfully flags half of all popular tracks while using a fraction of the memory.